## Test GPS Ephemeris Fit

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os

# my modules
from src.keplarian_ephemeris import *
from src.orbit_manager import *

In [ ]:
basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/Ephemeris/"
orbm_save_dir = basedir + "data/orbits"
figdir = basedir + "figures/gps_ephemeris_analysis"

if not os.path.exists(figdir):
    os.makedirs(figdir)

In [ ]:
t_hrs = 8
dt = 1


def get_initial_state(orbit_type, svid, M0_deg=None):
    # a, e, i, Omega, w, M0
    svoe = np.zeros((5, 6))
    # LCRNS
    if orbit_type == "LCRNS":
        t0_tai = pnt.convert_time(
            pnt.gregorian_to_time(2027, 3, 1, 0, 0, 0), pnt.UTC, pnt.TAI
        )
        svoe[0] = np.array(
            [11315.936501, 0.691982, 59.373229, 321.019197, 92.494031, 0.000000]
        )
        svoe[1] = np.array(
            [11317.948675, 0.691982, 58.951732, 320.997768, 92.505016, 180.000000]
        )
        svoe[2] = np.array(
            [11305.413654, 0.691982, 52.733096, 81.148790, 92.062891, 140.049207]
        )
        svoe[3] = np.array(
            [11326.302154, 0.691982, 52.513419, 81.138818, 92.068945, 195.992393]
        )
        svoe[4] = np.array(
            [11307.882863, 0.691982, 56.310396, 204.889626, 85.444071, 164.007607]
        )
        svoe[:, 2:] = np.deg2rad(svoe[:, 2:])
        svoe[:, 0] *= 1e3  # km to m
        coe_pa = svoe[svid]
        if M0_deg is not None:
            coe_pa[5] = np.deg2rad(M0_deg)
        rv0_pa0 = pnt.classical_to_cart(coe_pa, pnt.GM_MOON)  # In PA frame
        rv0_mci = convert_pa2ci(t0_tai, rv0_pa0, rotate_only=True)
    elif orbit_type == "Moonlight":
        t0_tai = pnt.convert_time(
            pnt.gregorian_to_time(2027, 1, 1, 0, 0, 0), pnt.TDB, pnt.TAI
        )
        a = 9748.14e3  # meters
        ecc = 0.70
        inc = np.deg2rad(48.04)
        w = np.deg2rad(123.60)
        Omega = np.deg2rad(89.49)
        # M0 = np.deg2rad(pnt.true_to_mean_anomaly(np.deg2rad(90.0), ecc))
        if M0_deg is None:
            M0_deg = pnt.true_to_mean_anomaly(np.deg2rad(90.0), ecc)
        else:
            M0 = np.deg2rad(M0_deg)
        coe = np.array([a, ecc, inc, Omega, w, M0])

        rv0_mci = pnt.classical_to_cart(coe, pnt.GM_MOON)  # In CI frame
        rv0_pa = convert_ci2pa(t0_tai, rv0_mci, rotate_only=True)

    elif orbit_type == "LNSS":
        t0_tai = pnt.convert_time(
            pnt.gregorian_to_time(2027, 1, 1, 0, 0, 0), pnt.TDB, pnt.TAI
        )
        a = 6541.4e3  # meters
        ecc = 0.60
        inc = np.deg2rad(56.2)
        w = np.deg2rad(90.0)
        Omega = np.deg2rad(0.0)
        if M0_deg is None:
            M0_deg = 0.0
        else:
            M0 = np.deg2rad(M0_deg)
        coe_op = np.array([a, ecc, inc, Omega, w, M0])
        rv0_op = pnt.classical_to_cart(coe_op, pnt.GM_MOON)
        rv0_mci = pnt.convert_frame(
            t0_tai, rv0_op, pnt.MOON_OP, pnt.MOON_CI
        )  # In CI frame
        rv0_pa = convert_ci2pa(t0_tai, rv0_mci, rotate_only=True)

    return t0_tai, rv0_mci

### Propagate

In [ ]:
# dynamics
add_earth = True
add_sun = True
sphm = [20, 20]  # Spherical harmonic model degree and order

dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-12, reltol=1e-12))
dyn.add_body(pnt.Body.Moon(sphm[0], sphm[1]))
if add_earth:
    dyn.add_body(pnt.Body.Earth())
if add_sun:
    dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(30.0)  # propagation timestep

In [ ]:
# propagate
def get_orbit(orbit_type, svid=0, M0_deg=None):
    t0_tai, rv0_mci = get_initial_state(orbit_type, svid, M0_deg)

    tend = t_hrs * 3600.0  # hours to seconds
    tspan = np.linspace(0.0, tend, int(tend / dt) + 1)  # every minute
    t_tai = tspan + t0_tai
    pnt.set_lupnt_epoch(0)
    rv_prop_mci = dyn.propagate(rv0_mci, t_tai)
    coe_prop_mci = pnt.cart_to_classical(rv_prop_mci, pnt.GM_MOON)

    # convert to PA frame
    t_tai = tspan + t0_tai
    rv_prop_pa = convert_ci2pa(t_tai, rv_prop_mci, rotate_only=True)
    rv_prop_pa_w = convert_ci2pa(t_tai, rv_prop_mci, rotate_only=False)
    coe_prop_pa = pnt.cart_to_classical(rv_prop_pa, pnt.GM_MOON)  # In PA frame

    return t_tai, tspan, rv_prop_pa, rv_prop_pa_w

## Fitting

In [ ]:
from src.gps_ephemeris import GPSEphemeris

t_fit_mins = np.linspace(10, 240, 24)  # Fit durations from 10 minutes to 4 hours
orbit_types = ["LCRNS"]
M0_degs = np.linspace(0, 360, 4, endpoint=False)  # M0 values from 0 to 360 degrees
gps_ephem = GPSEphemeris(body=pnt.MOON)
pos_error_p99 = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
pos_error_p95 = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
pos_error_rms = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
vel_error_p99 = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
vel_error_p95 = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
vel_error_rms = np.zeros((len(orbit_types), len(M0_degs), len(t_fit_mins)))
verbose = False
use_scipy_fit = True

for idx, orbit_type in enumerate(orbit_types):
    for j, M0_deg in enumerate(M0_degs):
        t_tai, tspan, rv_prop_pa, rv_prop_pa_w = get_orbit(
            orbit_type, svid=0, M0_deg=M0_deg
        )
        print("=================================================================")
        print(
            "Analyzing orbit type: {} with M0 = {} degrees".format(orbit_type, M0_deg)
        )
        print("==================================================================")

        for i, t_fit_min in enumerate(t_fit_mins):
            t_fit = t_fit_min * 60.0  # minutes to seconds
            t_fit_idx = np.where(tspan <= t_fit)[0]
            t_fit_span = tspan[t_fit_idx]
            rv_fit_pa = rv_prop_pa[t_fit_idx]
            rv_fit_pa_w = rv_prop_pa_w[t_fit_idx]

            if use_scipy_fit:
                ephem_opt = gps_ephem.fit_coeff_scipy(
                    t_fit_span,
                    rv_fit_pa,
                    loss="soft_l1",
                    f_scale=100.0,
                    method="trf",
                    max_nfev=200,
                    x_scale="jac",
                    verbose=verbose,
                )
            else:
                ephem_opt = gps_ephem.fit_coeff(
                    t_fit_span,
                    rv_fit_pa,
                    max_iter=20,
                    tol=1e-6,
                    fd_rel_step=1e-7,
                    fd_abs_floor=1e-10,
                    damping=0.0,
                    verbose=verbose,
                )
            ephem_opt_dict = gps_ephem.ephem2dict(ephem_opt)

            diff_y, df_pos, df_vel = gps_ephem.eval_fit_error(
                t_fit_span,
                rv_fit_pa,
                rv_fit_pa_w,
                ephem_opt,
                use_grad_for_velfit=True,
                print_stats=verbose,
            )
            pos_error_p99[idx, j, i] = np.percentile(
                np.linalg.norm(diff_y[:, :3], axis=1), 99.7
            )
            pos_error_p95[idx, j, i] = np.percentile(
                np.linalg.norm(diff_y[:, :3], axis=1), 95
            )
            pos_error_rms[idx, j, i] = np.sqrt(
                np.mean(np.linalg.norm(diff_y[:, :3], axis=1) ** 2)
            )
            vel_error_p99[idx, j, i] = (
                np.percentile(np.linalg.norm(diff_y[:, 3:], axis=1), 99.7) * 1e3
            )  # Convert m/s to mm/s
            vel_error_p95[idx, j, i] = (
                np.percentile(np.linalg.norm(diff_y[:, 3:], axis=1), 95) * 1e3
            )  # Convert m/s to mm/s
            vel_error_rms[idx, j, i] = (
                np.sqrt(np.mean(np.linalg.norm(diff_y[:, 3:], axis=1) ** 2)) * 1e3
            )  # Convert m/s to mm/s
            print(
                "Fit duration: {:>6.1f} min, 95% position error: {:>10.3f} m  and 95% velocity error: {:>10.3f} mm/s".format(
                    t_fit_min, pos_error_p95[idx, j, i], vel_error_p95[idx, j, i]
                )
            )
    print(" ")

In [ ]:
# Plotting
import matplotlib.pyplot as plt

t_fit_mins = np.linspace(10, 240, 24)  # Fit durations from 10 minutes to 4 hours
fontsize = 14
fontsize_legend = 12
fontsize_ticks = 14

plt.figure(figsize=(8, 5))
for idx, orbit_type in enumerate(orbit_types):
    for j, M0_deg in enumerate(M0_degs):
        plt.plot(
            t_fit_mins,
            pos_error_p99[idx, j, :],
            marker="o",
            label="M0={:.0f} deg".format(M0_deg),
        )
plt.xlabel("Fitting Duration (minutes)", fontsize=fontsize)
plt.ylabel("Position Error (95th Percentile, m)", fontsize=fontsize)
plt.title("GPS Ephemeris Fit Position Error vs Fitting Duration", fontsize=fontsize)
plt.grid()
plt.xticks(np.arange(0, 250, 30), fontsize=fontsize_ticks)
plt.yscale("log")
plt.axhline(13.43, color="black", linestyle="--", label="13.43 m (LCRNS requirement)")
plt.legend(fontsize=fontsize_legend)
plt.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
plt.tight_layout()
plt.savefig(os.path.join(figdir, "gps_ephemeris_fit_error.pdf"), dpi=300)
plt.show()

# Velocity error plot
plt.figure(figsize=(8, 5))
for idx, orbit_type in enumerate(orbit_types):
    for j, M0_deg in enumerate(M0_degs):
        plt.plot(
            t_fit_mins,
            vel_error_p99[idx, j, :],
            marker="o",
            label="M0={:.0f} deg".format(M0_deg),
        )
plt.xlabel("Fitting Duration (minutes)", fontsize=fontsize)
plt.ylabel("Velocity Error (95th Percentile, mm/s)", fontsize=fontsize)
plt.title("GPS Ephemeris Fit Velocity Error vs Fitting Duration", fontsize=fontsize)
plt.grid()
plt.yscale("log")
plt.xticks(np.arange(0, 250, 30), fontsize=fontsize_ticks)
plt.axhline(1.2, color="black", linestyle="--", label="1.2 mm/s (LCRNS requirement)")
plt.legend(fontsize=fontsize_legend)
plt.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
plt.tight_layout()
plt.savefig(os.path.join(figdir, "gps_ephemeris_fit_velocity_error.pdf"), dpi=300)
plt.show()

## Compare Fitting Error Changes in Different Methods

In [ ]:
from src.cartesian_ephemeris import CartesianEphemeris

methods = ["gps", "cheby", "kep_cheby", "kep_cheby_fourier"]
orbit_type = "LCRNS"
M0_degs = [0, 180]
t_fit_mins = np.linspace(
    30, 480, 16, endpoint=True
)  # Fit durations from 10 minutes to 4 hours
order = 12

# chebyshev ephemeris
cheby_ephem = CartesianEphemeris(
    order=order,
    use_kep=False,
    use_rsw=False,
    use_fourier=False,
    use_meq=False,
    poly_type="chebyshev",
    convert_to_coe=True,
    body=pnt.MOON,
)
kep_cheby_ephem = CartesianEphemeris(
    order=order,
    use_kep=True,
    use_rsw=False,
    use_fourier=False,
    use_meq=False,
    poly_type="chebyshev",
    convert_to_coe=True,
    body=pnt.MOON,
)
# cheby-fourier ephemeris
kep_cheby_fourier_ephem = CartesianEphemeris(
    order=order,
    use_kep=True,
    use_rsw=False,
    use_fourier=True,
    use_meq=False,
    poly_type="chebyshev",
    convert_to_coe=True,
    body=pnt.MOON,
)

pos_error_p99_method = np.zeros((len(methods), len(M0_degs), len(t_fit_mins)))
vel_error_p99_method = np.zeros((len(methods), len(M0_degs), len(t_fit_mins)))
verbose = False

for k, method in enumerate(methods):
    print("Method: {}".format(method))
    for j, M0_deg in enumerate(M0_degs):
        t_tai, t_span, rv_prop_pa, rv_prop_pa_w = get_orbit(
            orbit_type, svid=0, M0_deg=M0_deg
        )
        print("=================================================================")
        print(
            "Analyzing orbit type: {} with M0 = {} degrees".format(orbit_type, M0_deg)
        )
        print("==================================================================")

        for i, t_fit_min in enumerate(t_fit_mins):
            t_fit = t_fit_min * 60.0  # minutes to seconds
            t_fit_idx = np.where(tspan <= t_fit)[0]
            t_fit_span = tspan[t_fit_idx]
            rv_fit_pa = rv_prop_pa[t_fit_idx]
            rv_fit_pa_w = rv_prop_pa_w[t_fit_idx]

            if method == "gps":
                ephem_opt = gps_ephem.fit_coeff_scipy(
                    t_fit_span,
                    rv_fit_pa,
                    loss="soft_l1",
                    f_scale=100.0,
                    method="trf",
                    max_nfev=200,
                    x_scale="jac",
                    verbose=verbose,
                )
                diff_y, df_pos, df_vel = gps_ephem.eval_fit_error(
                    t_fit_span,
                    rv_fit_pa,
                    rv_fit_pa_w,
                    ephem_opt,
                    use_grad_for_velfit=True,
                    print_stats=verbose,
                )
            elif method == "cheby":
                ephem_opt = cheby_ephem.fit(
                    t_fit_span, rv_fit_pa, fit_obj="lsq", print_result=False
                )
                diff_y, df_pos, df_vel = cheby_ephem.eval_fit_error(
                    t_fit_span,
                    rv_fit_pa,
                    rv_fit_pa_w,
                    ephem_opt,
                    use_grad_for_velfit=True,
                    print_stats=verbose,
                )
            elif method == "kep_cheby":
                ephem_opt = kep_cheby_ephem.fit(
                    t_fit_span, rv_fit_pa, fit_obj="lsq", print_result=False
                )
                diff_y, df_pos, df_vel = kep_cheby_ephem.eval_fit_error(
                    t_fit_span,
                    rv_fit_pa,
                    rv_fit_pa_w,
                    ephem_opt,
                    use_grad_for_velfit=True,
                    print_stats=verbose,
                )
            elif method == "kep_cheby_fourier":
                ephem_opt = kep_cheby_fourier_ephem.fit(
                    t_fit_span, rv_fit_pa, fit_obj="lsq", print_result=False
                )
                diff_y, df_pos, df_vel = kep_cheby_fourier_ephem.eval_fit_error(
                    t_fit_span,
                    rv_fit_pa,
                    rv_fit_pa_w,
                    ephem_opt,
                    use_grad_for_velfit=True,
                    print_stats=verbose,
                )

            pos_error_p99_method[k, j, i] = np.percentile(
                np.linalg.norm(diff_y[:, :3], axis=1), 99.7
            )
            vel_error_p99_method[k, j, i] = (
                np.percentile(np.linalg.norm(diff_y[:, 3:], axis=1), 99.7) * 1e3
            )  # Convert m/s to mm/s
            print(
                "Fit duration: {:>6.1f} min, 99% position error: {:>10.3f} m  and 99% velocity error: {:>10.3f} mm/s".format(
                    t_fit_min,
                    pos_error_p99_method[k, j, i],
                    vel_error_p99_method[k, j, i],
                )
            )

In [ ]:
# Plotting
method_labels = [
    "GPS Ephemeris",
    "Chebyshev({})".format(order),
    "Kepler+Chebyshev({})".format(order),
    "Kepler+Chebyshev({})+Fourier".format(order),
]
fontsize = 14
fontsize_legend = 12
fontsize_ticks = 12

# M = 0 deg Position
for j, M0_deg in enumerate(M0_degs):
    fig, ax = plt.subplots(figsize=(6, 5))
    for k, method in enumerate(methods):
        plt.plot(
            t_fit_mins,
            pos_error_p99_method[k, j, :],
            marker="o",
            label="{0}".format(method_labels[k]),
        )
    plt.xlabel("Fitting Duration (minutes)", fontsize=fontsize)
    plt.ylabel("Position Error (99th Percentile, m)", fontsize=fontsize)
    plt.title(
        "Ephemeris Fit Position Error vs Fitting Duration for M0={:.0f} deg".format(
            M0_deg
        ),
        fontsize=fontsize,
    )
    plt.grid()
    plt.yscale("log")
    plt.axhline(
        13.43, color="black", linestyle="--", label="13.43 m (Lunar requirement)"
    )
    # plt.legend(fontsize=fontsize_legend)
    plt.xlim(0, 480)
    plt.xticks(np.arange(0, 481, 30))
    # ticks font size
    ax.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
    plt.legend(fontsize=fontsize_legend)
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            figdir,
            "ephemeris_fit_position_error_comparison_M0_{:.0f}_deg.pdf".format(M0_deg),
        ),
        dpi=300,
    )
    plt.show()

    # M = 0 deg Velocity
    fig, ax = plt.subplots(figsize=(6, 5))
    for k, method in enumerate(methods):
        plt.plot(
            t_fit_mins,
            vel_error_p99_method[k, j, :],
            marker="o",
            label="{0}".format(method_labels[k]),
        )
    plt.xlabel("Fitting Duration (minutes)", fontsize=fontsize)
    plt.ylabel("Velocity Error (99th Percentile, mm/s)", fontsize=fontsize)
    plt.title(
        "Ephemeris Fit Velocity Error vs Fitting Duration for M0={:.0f} deg".format(
            M0_deg
        ),
        fontsize=fontsize,
    )
    plt.grid()
    plt.yscale("log")
    plt.axhline(
        1.2, color="black", linestyle="--", label="1.2 mm/s (LCRNS requirement)"
    )
    # plt.legend(fontsize=fontsize_legend)
    plt.xlim(0, 480)
    plt.xticks(np.arange(0, 481, 30))
    # ticks font size
    ax.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
    plt.legend(fontsize=fontsize_legend)
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            figdir,
            "ephemeris_fit_velocity_error_comparison_M0_{:.0f}_deg.pdf".format(M0_deg),
        ),
        dpi=300,
    )
    plt.show()